# Cuaderno 8 A2 — CNN 2D (MNIST) y CNN 3D (Conv3D) en un solo cuaderno

Corre **todo de una vez** (`Entorno de ejecución → Ejecutar todas`) y entrena los dos modelos, cada uno en su problema, guardando los dos en tu Drive:

- `mnist_conv2d.keras` → dígitos MNIST, entrada `(28, 28, 1)`, 2 capas **Conv2D**
- `esfera_cubo_conv3d.keras` → esfera vs cubo, entrada `(16, 16, 16, 1)`, 3 capas **Conv3D**

Ambos quedan en `MyDrive/modelos_cnn/`.

> **Ojo:** los dos modelos NO se comparan entre sí. Resuelven problemas distintos, con entradas de forma distinta: uno mira una foto de un dígito, el otro mira un volumen 3D. La tabla final muestra la precisión de cada uno **en su propio conjunto de prueba** — sirve para saber que cada modelo quedó bien, no para decir "cuál es mejor".

Este cuaderno es el que se corre; los dos cuadernos originales (`Números Modo 2` y `Conv3D Esfera vs Cubo`) se quedan aparte como material de la clase.

## 0. Importaciones y conexión con Drive

Una sola vez para todo el cuaderno: los dos modelos se guardan en `MyDrive/modelos_cnn/`.

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow:", tf.__version__)

from google.colab import drive
drive.mount('/content/drive')
CARPETA = '/content/drive/MyDrive/modelos_cnn'
os.makedirs(CARPETA, exist_ok=True)
print("Los modelos se guardan en:", CARPETA)

TensorFlow: 2.21.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Los modelos se guardan en: /content/drive/MyDrive/modelos_cnn


In [8]:
#Verificar gpu 
!nvidia-smi


Fri Oct  9 19:50:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             28W /   70W |     639MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---
# Sección A — CNN 2D con MNIST (dígitos escritos a mano)

## A1. Cargar y preparar MNIST (60.000 entrenamiento / 10.000 prueba)

Se usa el MNIST que trae Keras (`keras.datasets.mnist`), que no depende de `tensorflow_datasets`.

In [9]:
# El MNIST que trae Keras: ya viene con TensorFlow, no descarga nada extra.
(X_train_m, y_train_m), (X_test_m, y_test_m) = keras.datasets.mnist.load_data()

X_train_m = (X_train_m.astype('float32') / 255.0)[..., None]   # (60000, 28, 28, 1)
X_test_m  = (X_test_m.astype('float32') / 255.0)[..., None]    # (10000, 28, 28, 1)

print("Entrenamiento:", X_train_m.shape, "| Prueba:", X_test_m.shape)

Entrenamiento: (60000, 28, 28, 1) | Prueba: (10000, 28, 28, 1)


## A2. Modelo 2D (la misma arquitectura del cuaderno del curso: Conv2D 32 y 64)

In [10]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2), padding="same"),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=10, activation='softmax')
])

modelo.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)
modelo.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │       147,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 168,746 (659.16 KB)

 Trainable params: 168,746 (659.16 KB)

 Non-trainable params: 0 (0.00 B)

## A3. Entrenar (early stopping) y guardar `mnist_conv2d.keras` en Drive

Como en el cuaderno original, se valida con el conjunto de prueba.

In [11]:
historial = modelo.fit(
    X_train_m, y_train_m,
    validation_data=(X_test_m, y_test_m),
    epochs=50, batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

print("Epocas ejecutadas:", len(historial.history['loss']))
print("Precision final entrenamiento: {:.4f}".format(historial.history['accuracy'][-1]))
print("Precision final validacion:    {:.4f}".format(historial.history['val_accuracy'][-1]))

modelo.save(f'{CARPETA}/mnist_conv2d.keras')
print("Guardado:", f'{CARPETA}/mnist_conv2d.keras')

Epoch 1/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - accuracy: 0.9528 - loss: 0.1534 - val_accuracy: 0.9820 - val_loss: 0.0524
Epoch 2/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9852 - loss: 0.0498 - val_accuracy: 0.9892 - val_loss: 0.0338
Epoch 3/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9887 - loss: 0.0352 - val_accuracy: 0.9893 - val_loss: 0.0324
Epoch 4/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9921 - loss: 0.0265 - val_accuracy: 0.9905 - val_loss: 0.0311
Epoch 5/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9935 - loss: 0.0202 - val_accuracy: 0.9897 - val_loss: 0.0285
Epoch 6/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9943 - loss: 0.0173 - val_accuracy: 0.9893 - val_loss: 0.0334
Epoch 7/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9963 - loss: 0.0121 - val_accuracy: 0.9873 - val_loss: 0.0433
Epoch 8/50
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9961 - loss: 0.0119 -

---
# Sección B — CNN 3D con Conv3D (esfera vs cubo, volúmenes sintéticos)

## B1. Generar los volúmenes 16×16×16 y separar entrenamiento / validación / prueba

In [12]:
def crear_volumen(etiqueta, rng, tam=16):
    coords = np.indices((tam, tam, tam)).astype(np.float32)
    centro = np.array([tam/2 + rng.uniform(-1, 1) for _ in range(3)], dtype=np.float32)
    dx, dy, dz = coords[0]-centro[0], coords[1]-centro[1], coords[2]-centro[2]
    if etiqueta == 0:  # esfera
        radio = rng.uniform(4.0, 5.2)
        mascara = dx**2 + dy**2 + dz**2 <= radio**2
    else:  # cubo
        lado = rng.uniform(4.0, 5.2)
        mascara = (np.abs(dx) <= lado) & (np.abs(dy) <= lado) & (np.abs(dz) <= lado)
    volumen = mascara.astype(np.float32)
    volumen += rng.normal(0, 0.06, volumen.shape).astype(np.float32)
    return np.clip(volumen, 0, 1)

rng = np.random.default_rng(42)
n = 1200
X = np.empty((n, 16, 16, 16, 1), dtype=np.float32)
y = np.empty(n, dtype=np.int32)
for i in range(n):
    y[i] = i % 2
    X[i, ..., 0] = crear_volumen(int(y[i]), rng)
orden = rng.permutation(n)
X, y = X[orden], y[orden]

n_train = int(len(X) * 0.70)
n_val = int(len(X) * 0.15)
X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test, y_test = X[n_train+n_val:], y[n_train+n_val:]
print("Forma de X:", X.shape)
print("Entrenamiento:", X_train.shape, "| Validacion:", X_val.shape, "| Prueba:", X_test.shape)

Forma de X: (1200, 16, 16, 16, 1)
Entrenamiento: (840, 16, 16, 16, 1) | Validacion: (180, 16, 16, 16, 1) | Prueba: (180, 16, 16, 16, 1)


## B2. Modelo 3D (Conv3D 16, 32 y 64 con MaxPooling3D)

In [13]:
modelo_3d = keras.Sequential([
    keras.Input(shape=(16, 16, 16, 1)),
    layers.Conv3D(16, 3, padding="same", activation="relu"),
    layers.MaxPooling3D(2),
    layers.Conv3D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling3D(2),
    layers.Conv3D(64, 3, padding="same", activation="relu"),
    layers.GlobalAveragePooling3D(),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.25),
    layers.Dense(1, activation="sigmoid")
])
modelo_3d.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo_3d.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv3d (Conv3D)                 │ (None, 16, 16, 16, 16) │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d (MaxPooling3D)    │ (None, 8, 8, 8, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 8, 8, 8, 32)    │        13,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_1 (MaxPooling3D)  │ (None, 4, 4, 4, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 4, 4, 4, 64)    │        55,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling3d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling3D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,777 (280.38 KB)

 Trainable params: 71,777 (280.38 KB)

 Non-trainable params: 0 (0.00 B)

## B3. Entrenar (early stopping) y guardar `esfera_cubo_conv3d.keras` en Drive

In [14]:
historial_3d = modelo_3d.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15, batch_size=16,
    callbacks=[EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)],
    verbose=1
)

loss_test, acc_test = modelo_3d.evaluate(X_test, y_test, verbose=0)
print("Epocas ejecutadas:", len(historial_3d.history['loss']))
print("Perdida en prueba:    {:.4f}".format(loss_test))
print("Precision en prueba:  {:.4f}".format(acc_test))

modelo_3d.save(f'{CARPETA}/esfera_cubo_conv3d.keras')
print("Guardado:", f'{CARPETA}/esfera_cubo_conv3d.keras')

# comprobacion: se puede volver a cargar desde Drive
modelo_3d_cargado = keras.models.load_model(f'{CARPETA}/esfera_cubo_conv3d.keras')
print("Recargado desde Drive, entrada esperada:", modelo_3d_cargado.input_shape)

Epoch 1/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 12s 123ms/step - accuracy: 0.7869 - loss: 0.4208 - val_accuracy: 1.0000 - val_loss: 0.0072
Epoch 2/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.0075 - val_accuracy: 1.0000 - val_loss: 3.9305e-04
Epoch 3/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0015 - val_accuracy: 1.0000 - val_loss: 9.3744e-05
Epoch 4/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 6.1984e-04 - val_accuracy: 1.0000 - val_loss: 1.4597e-05
Epoch 5/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 1.6073e-04 - val_accuracy: 1.0000 - val_loss: 9.2655e-06
Epoch 6/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 6.6406e-04 - val_accuracy: 1.0000 - val_loss: 6.4245e-06
Epoch 7/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 1.2664e-04 - val_accuracy: 1.0000 - val_loss: 3.4904e-06
Epoch 8/15
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss

---
# Sección C — Resumen: los dos modelos, cada uno en su prueba

Carga los dos `.keras` **desde el Drive** (no de la memoria) y muestra la precisión de cada uno. Si los dos números salen altos, los dos modelos quedaron bien guardados y sirven para la app. Esta sección usa datos que ya están en memoria, así que se corre después de A y B.

In [15]:
for archivo in ('mnist_conv2d.keras', 'esfera_cubo_conv3d.keras'):
    assert os.path.exists(f'{CARPETA}/{archivo}'), \
        f"No existe {archivo} en Drive: corre primero las secciones A y B en esta misma sesion."

m2 = keras.models.load_model(f'{CARPETA}/mnist_conv2d.keras')
m3 = keras.models.load_model(f'{CARPETA}/esfera_cubo_conv3d.keras')

_, acc2 = m2.evaluate(X_test_m, y_test_m, verbose=0)
_, acc3 = m3.evaluate(X_test, y_test, verbose=0)

resumen = [
    ("mnist_conv2d.keras",       "digitos MNIST (28x28x1)",     "2 x Conv2D", 10, acc2),
    ("esfera_cubo_conv3d.keras", "esfera vs cubo (16x16x16x1)", "3 x Conv3D",  2, acc3),
]
print(f"{'archivo':<28}{'problema':<28}{'arquitectura':<14}{'clases':>7}{'accuracy':>10}")
for nombre, problema, arq, cl, acc in resumen:
    print(f"{nombre:<28}{problema:<28}{arq:<14}{cl:>7}{acc:>10.4f}")
print()
print("Recordatorio: son dos problemas distintos, la tabla NO dice cual es mejor.")
print("Para la app: mnist_conv2d.keras come una imagen 28x28 en gris;")
print("esfera_cubo_conv3d.keras come un volumen 16x16x16.")

archivo                     problema                    arquitectura   clases  accuracy
mnist_conv2d.keras          digitos MNIST (28x28x1)     2 x Conv2D         10    0.9897
esfera_cubo_conv3d.keras    esfera vs cubo (16x16x16x1) 3 x Conv3D          2    1.0000

Recordatorio: son dos problemas distintos, la tabla NO dice cual es mejor.
Para la app: mnist_conv2d.keras come una imagen 28x28 en gris;
esfera_cubo_conv3d.keras come un volumen 16x16x16.
